# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashishpal003/flyrank_ml_intern/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Freestyle B — Decline early-warning.** Predict which currently-healthy pages will lose search traffic in the next ~30 days, so an editor can review them *before* the drop.

Why this one:

- It targets the decision FlyRank actually makes every week (which pages to review), not a side question.
- The label is a genuine **observed future outcome** — measured traffic in a later window — so the model learns the world, not a rule.
- There is a **real baseline to beat**: FlyRank's existing "stale + still visible" rule, scorable on the exact same pages.
- Momentum/growth prediction is kept as a *possible* second output only if ahead of schedule. Recovery prediction is out of scope: it is partly causal (recovery is often driven by an editor's refresh) and needs an experiment we do not have.

In [1]:
from pathlib import Path
import pandas as pd

CSV = Path("../../data/raw/content_refresh_anonymized.csv")
if not CSV.exists():
    CSV = Path("data/raw/content_refresh_anonymized.csv")  # when run from repo root
df = pd.read_csv(CSV)
print(f"loaded {len(df):,} rows x {df.shape[1]} columns from {CSV.name}")

loaded 30,000 rows x 44 columns from content_refresh_anonymized.csv


## 2. The question: decision, action, cost of a wrong call

### What decision this improves

**Primary:** *"Which pages should this client's content strategist put on the refresh/review shortlist this cycle, before the decline is visible in standard reporting?"*

Today FlyRank reacts: the drop surfaces in the GSC dashboard and in the `trend_direction` bucket **only once it is already a measured 30-day decline**. By then the page has lost weeks of traffic and rankings are stickier to recover. This project moves the trigger earlier — flagging pages while they still look "stable" but show precursor patterns.

**Secondary decisions it supports:**

- *Capacity planning* — "how many pages across my book of clients are entering the risk zone this month?" informs where to add writer hours.
- *Client prioritization* — a client-level rollup of at-risk pages shows which accounts need a check-in.
- *Baseline audit* — comparing against FlyRank's stale-visible rule on the same pages shows where that rule is blind (e.g. pages that decline without being stale).

**What it does _not_ decide:** *what* to do to a flagged page (refresh / expand / consolidate / rewrite title / leave) — that stays with the editor, informed by reason codes — and whether any intervention will actually work.

### Who acts on the output

**Primary actor — the FlyRank content strategist / editor for a client account.** Concretely they:

1. Open the weekly ranked queue for their client(s).
2. For each page near the top: open it, read the reason codes and the recent trend, and make a call — schedule a refresh, expand it, flag a title/meta rewrite, or mark "watchlist / no action."
3. Dismiss false alarms (this feedback also becomes evaluation signal).

**Secondary actor — the account manager / team lead**, who uses the client-level count ("12 pages at risk this month for Client X, up from 4") for check-ins and resourcing.

**Not an actor:** the end client (sees outcomes, not the queue) and any automated system — nothing auto-edits a page. Human-in-the-loop by design.

### Cost of a wrong recommendation (asymmetric)

**False positive — flagged as at-risk, doesn't decline:**

- ~1–2 editor hours reviewing/refreshing a page that was fine (the refresh itself rarely hurts, but the capacity is spent).
- Opportunity cost: that hour didn't go to a page that *was* sliding.
- **Trust cost** — too many early false alarms and editors stop opening the queue; a decision aid nobody trusts is dead. Mitigated by a conservative K, reason codes, and a confidence tier.
- Bounded, recoverable, measured in hours.

**False negative — declines, never flagged:**

- The page bleeds organic traffic for weeks before the standard bucket catches it — cumulative lost impressions/clicks over that lag.
- Slipped rankings cost more editor effort to win back than an early refresh would have.
- Client-facing: a visible dip in the monthly report.
- Larger, compounding — and it is the exact failure the project exists to reduce.

**Third, sneakier failure — the model games the label** by flagging tiny-volume pages (60 -> 30 impressions is "-50%" but meaningless). The queue fills with noise even at high "precision." Guarded by the volume floor in the eligibility rule and by hand-reviewing the top 20 every iteration.

**Implication for method:** misses are costlier than false alarms, but review capacity is fixed — so the metric is **precision@K** (fixed budget) reported *alongside* recall@K and the base rate, and the model is tuned as a **ranking**, not a hard yes/no.

### Why a plain rule / dashboard isn't enough

FlyRank's current rule already works at the top of the list but runs out where signals get many, tangled, and shifting. Near-future decline in this portfolio depends on the interaction of position drift, query-mix narrowing, freshness, seasonality, and traffic shape — messy enough to be worth learning, and shifting enough over 17 months that fixed thresholds go stale. A dashboard shows *what happened*; this predicts *what's about to*.

In [2]:
# Sketch of the fixed-budget framing this section commits to (numbers finalized in ML-03/04).
frame = pd.DataFrame(
    [
        ("grain", "one row = one (content page x decision date)"),
        ("feature window", "[decision_date - 90d, decision_date)"),
        ("label window", "[decision_date, decision_date + 30d]"),
        ("task type", "binary classification -> probability -> ranking"),
        ("primary metric", "precision@K (recall@K, avg precision, base rate alongside)"),
        ("baseline to beat", "FlyRank stale-visible rule, same pages / metric / split"),
        ("validation", "client-grouped AND time-forward; sealed final month"),
    ],
    columns=["piece", "provisional choice"],
)
print(frame.to_string(index=False))

           piece                                         provisional choice
           grain               one row = one (content page x decision date)
  feature window                       [decision_date - 90d, decision_date)
    label window                       [decision_date, decision_date + 30d]
       task type            binary classification -> probability -> ranking
  primary metric precision@K (recall@K, avg precision, base rate alongside)
baseline to beat    FlyRank stale-visible rule, same pages / metric / split
      validation        client-grouped AND time-forward; sealed final month


## 3. Quick look at the data (2-3 real numbers)

Three numbers, computed live from the starter CSV, that make this lane worth the next 7 weeks. All are **observed** slices of one 30k-row snapshot — directional motivation, not the forward-label base rate (that is set in ML-03).

1. **The problem is common.** Share of pages with `trend_direction == "down"` — decline is a large, real slice of the portfolio, not a rare edge case.
2. **Clicks can't carry the label; impressions can.** Share of pages clearing a `impressions_prev_30d >= 100` bar vs a `clicks_prev_30d >= 20` bar. Clicks are far too sparse to define a per-page outcome — this is why the label is **impressions primary, clicks confirmatory**.
3. **There is real exposure at stake.** Median and total `impressions_90d` on the pages currently trending down — the traffic sitting on pages that need attention.

No client names, no row-level dumps.

In [3]:
n = len(df)

# 1. Decline is common in this snapshot.
down = df["trend_direction"] == "down"
pct_down = 100 * down.mean()

# 2. Impressions can anchor a per-page label; clicks are too sparse.
pct_imp_100 = 100 * (df["impressions_prev_30d"] >= 100).mean()
pct_clk_20 = 100 * (df["clicks_prev_30d"] >= 20).mean()

# 3. Exposure sitting on the declining pages.
imp90_down_median = df.loc[down, "impressions_90d"].median()
imp90_down_total = df.loc[down, "impressions_90d"].sum()

print(f"rows in snapshot:                          {n:,}")
print(f"1. pages with trend_direction == 'down':   {pct_down:.1f}%")
print(f"2. pages with impressions_prev_30d >= 100: {pct_imp_100:.1f}%")
print(f"   pages with clicks_prev_30d >= 20:       {pct_clk_20:.1f}%")
print(f"   -> impressions bar is ~{pct_imp_100/pct_clk_20:.0f}x more populated than the clicks bar")
print(f"3. impressions_90d on 'down' pages -- median: {imp90_down_median:,.0f}   total: {imp90_down_total:,.0f}")

rows in snapshot:                          30,000
1. pages with trend_direction == 'down':   54.2%
2. pages with impressions_prev_30d >= 100: 60.0%
   pages with clicks_prev_30d >= 20:       5.9%
   -> impressions bar is ~10x more populated than the clicks bar
3. impressions_90d on 'down' pages -- median: 961   total: 79,994,363


**Reading the numbers.** About half the snapshot is already bucketed `down`, so decline is the portfolio's normal state, not a tail case — an early-warning queue has plenty to rank. Only a small single-digit share of pages clear even a modest 20-clicks-per-30-days bar, versus a solid majority for 100 impressions, so a clicks-only label would be undefined for most pages; impressions carry it and clicks confirm direction. And the pages trending down carry a large median volume each and tens of millions of impressions in aggregate — the exposure a miss lets bleed is substantial. (These are one-snapshot descriptive slices; the forward, out-of-sample decline base rate is defined and measured in ML-03.)

## 4. Careful words: what I can and can't claim

### What the work WILL be able to say

- **Observed, out-of-sample:** "On clients and a calendar month the model never saw in training, pages it ranked in the top K declined at rate X — versus a base rate of Y and FlyRank's current stale-visible rule at Z."
- **Observed lead time:** "For the true declines it caught, the model flagged them a median of N days before the standard 30-day trend bucket would have." (only if it holds up)
- **Directional / association:** "In this portfolio, the signals most associated with a near-future impressions decline were [position drift, narrowing query spread, falling days-with-impressions, ...]." Association, not mechanism.
- **Decision-support:** "Used as a weekly review queue under a fixed editor budget, this ranking would concentrate attention on higher-risk pages than the existing rule."
- **Honest uncertainty:** "Performance varies widely by client (per-fold precision@K ranged A-B); it is weakest for clients with short history and flat, low-volume pages."
- **Scoped:** "One content operation, ~100 pseudonymized clients, ~17 months, one snapshot — a decision aid for FlyRank's editors, evaluated on FlyRank's own data."

### What the work will NEVER say

- No: "This page *will* decline." — it is a probability over a population.
- No: "Refreshing this page will prevent the decline / cause recovery." — no experiment, no causal design. The model predicts decline; it says nothing about whether an intervention works.
- No: "We predicted / reverse-engineered Google's ranking algorithm." — portfolio-specific correlation on downstream metrics, not knowledge of Google's system.
- No: "Feature X *causes* traffic loss." — association only; seasonality, market, and cannibalization are not controlled.
- No: "This generalizes to SEO in general / any website." — one operation, one period.
- No: "94% accurate" with no base rate beside it — banned by the honest-claims skill.
- No: any client name, domain, URL, page title, or real search query.
- No: "FlyRank's existing rule is bad." — constructive framing only: "where a learned ranking adds signal the rule doesn't capture."

In [4]:
# Leakage red list this frame commits to (enforced from ML-03 on).
red_list = [
    "trend_direction, trend_pct  (the label is derived from these)",
    "any *_last_30d / impressions_90d-style column whose window overlaps the label window",
    "   -> only *_prev_30d-type columns are window-safe features",
    "FlyRank product flags (health_score, stale-visible, ...) as features",
    "any signal of an edit made DURING the label window",
]
print("LEAKAGE RED LIST")
for item in red_list:
    print(" -", item)

LEAKAGE RED LIST
 - trend_direction, trend_pct  (the label is derived from these)
 - any *_last_30d / impressions_90d-style column whose window overlaps the label window
 -    -> only *_prev_30d-type columns are window-safe features
 - FlyRank product flags (health_score, stale-visible, ...) as features
 - any signal of an edit made DURING the label window


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Section 3's numbers are computed live from the CSV — nothing hardcoded
- [x] The one-paragraph frame reads with every blank filled honestly
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.